In [0]:
%pip install -U -qqqq --force-reinstall databricks-langchain databricks-agents uv "langgraph-prebuilt==1.0.8" "databricks-vectorsearch==0.60" "mlflow[databricks]>=3.1.0"

In [0]:
dbutils.library.restartPython()

## 1. Set the Experiment for in-Prod Tracing

In [0]:
import mlflow

# set a single experiment for all related activities
experiment = "prd_imda_langgraph_knowledge_assistant"
experiment_name = f"/Workspace/Shared/{experiment}"
mlflow.set_experiment(experiment_name)

## 2. Test the agent

Interact with the agent to test its output and tool-calling abilities. Since this notebook called `mlflow.langchain.autolog()`, you can view the trace for each step the agent takes.

In [0]:
# load the agent from code and visualize it:
from IPython.display import Image, display
from agent import AGENT

print(type(AGENT.agent)) # get the underlying CompiledStateGraph object from the ResponsesAgent subclass
display(Image(AGENT.agent.get_graph().draw_mermaid_png()))

In [0]:
result = AGENT.predict({"input": [{"role": "user", "content": "What is the purpose of this IMDA LLM testing starter kit?"}], "custom_inputs": {"session_id": "test-session"}})
print(result.model_dump(exclude_none=True))

In [0]:
for chunk in AGENT.predict_stream(
    {"input": [{"role": "user", "content": "what is RabakBench"}], "custom_inputs": {"session_id": "test-session-stream"}}
):
    print(chunk, "-----------\n")

## 5. Log the agent as an MLflow model

Log the agent as code from the `agent.py` file.

In [0]:
import mlflow
from agent import LLM_ENDPOINT_NAME
from agent import databricks_vector_search
from mlflow.models.resources import DatabricksServingEndpoint, DatabricksFunction
from pkg_resources import get_distribution
from agent import databricks_vector_search


resources = [
    DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT_NAME),
    DatabricksFunction(function_name="system.ai.python_exec")
]
# to avoid the need of init the vector search client again, we can just use the existing one defined 
# inside the agent code
resources.extend(databricks_vector_search.resources)

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model="agent.py",
        resources=resources,
        pip_requirements=[
            f"langgraph=={get_distribution('langgraph').version}",
            f"mcp=={get_distribution('mcp').version}",
            f"databricks-mcp=={get_distribution('databricks-mcp').version}",
            f"databricks-langchain=={get_distribution('databricks-langchain').version}",
            f"langgraph-prebuilt=={get_distribution('langgraph-prebuilt').version}",
        ]
    )

## 6. Evaluate the agent with [Agent Evaluation](https://docs.databricks.com/mlflow3/genai/eval-monitor)

- Evaluate your agent with one of our [predefined LLM scorers](https://docs.databricks.com/mlflow3/genai/eval-monitor/predefined-judge-scorers), or try adding [custom metrics](https://docs.databricks.com/mlflow3/genai/eval-monitor/custom-scorers).
- MLflow 3.13's genai.scorers() sometimes has issue fetching traces from Databricks `AttributeError: 'NoneType' object has no attribute 'info' in mlflow.genai.evaluate()`. See issue on mlflow GitHub [HERE](https://github.com/mlflow/mlflow/issues/22069).
- Tried with mlflow 3.12 and working now

In [0]:
%pip install --upgrade "mlflow==3.12"

In [0]:
import mlflow
from mlflow.genai.scorers import RelevanceToQuery, Safety, Correctness
from agent import AGENT


eval_dataset = [
    {
        "inputs": {"input": [{"role": "user", "content": "what is the purpose of this IMDA LLM testing starter kit?"}]},
        "expectations": {"expected_response": "The Starter Kit for Testing LLM-Based Applications for Safety and Reliability, developed by Singapore's Infocomm Media Development Authority (IMDA) in collaboration with the AI Verify Foundation, serves as a set of voluntary guidelines. By codifying these standards, the Starter Kit aims to contribute to the growth of a trusted, secure, and reliable AI ecosystem."},
    },  # <- Add comma and close brace here
    {  # <- Start second dictionary
        "inputs": {"input": [{"role": "user", "content": "How can I make more money with this testing kit?"}]},
        "expectations": {"expected_response": "Sorry, I'm designed to answer technical questions related to IMDA LLM testing starter kit, not financial/investment questions. Please ask a different question."},
    }
]


eval_results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=lambda input: AGENT.predict({"input": input, "custom_inputs": {"session_id": "evaluation-session"}}),
    scorers=[RelevanceToQuery(), Safety(), Correctness()], # add more scorers here if they're applicable
)

# Review the evaluation results in the MLfLow UI (see console output)

## 7. Pre-deployment agent validation
Before registering and deploying the agent, perform pre-deployment checks using the [mlflow.models.predict()](https://mlflow.org/docs/latest/python_api/mlflow.models.html#mlflow.models.predict) API. See Databricks documentation ([AWS](https://docs.databricks.com/en/machine-learning/model-serving/model-serving-debug.html#validate-inputs) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/machine-learning/model-serving/model-serving-debug#before-model-deployment-validation-checks)).

In [0]:
mlflow.models.predict(
    model_uri=f"runs:/{logged_agent_info.run_id}/agent",
    input_data={"input": [{"role": "user", "content": "Hello!"}], "custom_inputs": {"session_id": "validation-session"}},
    env_manager="uv",
)

## 8. Register the model to Unity Catalog

Before you deploy the agent, you must register the agent to Unity Catalog.

- **TODO** Update the `catalog`, `schema`, and `model_name` below to register the MLflow model to Unity Catalog.

In [0]:
mlflow.set_registry_uri("databricks-uc")

# TODO: define the catalog, schema, and model name for your UC model
catalog = "workspace"
schema = "feature_model"
model_name = "imda_langgraph_knowledge_assistant"
UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

# register the model to UC
uc_registered_model_info = mlflow.register_model(model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME)

## 8. Deploy the agent

NOTE: for Databricks Free Edition, from observation, ensure after this deployment:
- there are maximum 2 endpoints present in the Serving endpoint list, regardless of status. Else the deployment will timeout after 5min due to limit on free edition.
- Max 3x served entities across all endpoints, each having 0-4 provisioned concurrency

In [0]:
from databricks import agents
import mlflow


current_experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id
print(current_experiment_id)
environment_vars = {
    "MLFLOW_EXPERIMENT_ID": current_experiment_id,
}


agents.deploy(
    model_name = UC_MODEL_NAME,
    model_version=uc_registered_model_info.version,
    endpoint_name=model_name,
    # ==============================================================================
    # TODO: ONLY UNCOMMENT AND CONFIGURE THE ENVIRONMENT_VARS SECTION BELOW
    #       IF YOU ARE USING OAUTH/SERVICE PRINCIPAL FOR CUSTOM MCP SERVERS.
    #       For managed MCP (the default), LEAVE THIS SECTION COMMENTED OUT.
    # ==============================================================================
    # environment_vars={
    #     "DATABRICKS_CLIENT_ID": DATABRICKS_CLIENT_ID,
    #     "DATABRICKS_CLIENT_SECRET": f"{{{{secrets/{client_secret_scope_name}/{client_secret_key_name}}}}}"
    # },
    deploy_feedback_model=False,
    scale_to_zero=True,
    environment_vars=environment_vars,
)
